First I combine the files

In [8]:
import os
import json
import re
import unicodedata
import pandas as pd

In [9]:
# --- FILE PATHS ---
# Adjust these paths if your files are in different folders
DATA_DIR = 'data'
INPUT_CSV = os.path.join(DATA_DIR, 'billboard_to_musicbrainz.csv')
GENRES_FILE = os.path.join(DATA_DIR, 'genres_FINAL.json')
OUTPUT_CSV = os.path.join(DATA_DIR, 'billboard_to_mbz_with_simplified_genres.csv')

# --- NORMALIZATION FUNCTIONS ---
def normalize_name(name):
    if pd.isna(name):
        return ''
    text = unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii')
    text = text.lower().strip()
    text = text.replace('&', ' and ')
    text = re.sub(r'\(.*?\)|\[.*?\]', ' ', text)
    text = re.sub(r'\b(feat|featuring|ft)\.?\b.*$', ' ', text)
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def strip_the_prefix(name):
    return re.sub(r'^the\s+', '', name).strip()

# --- EXECUTION ---
print("1. Loading files...")
df = pd.read_csv(INPUT_CSV)

# Remove the old 'Genres' column if it exists so we have a clean slate
if 'Genres' in df.columns:
    df = df.drop(columns=['Genres'])

with open(GENRES_FILE, 'r', encoding='utf-8') as f:
    raw_genres = json.load(f)

print("2. Prepping JSON genre data...")
normalized_genres = {}
normalized_no_the_genres = {}

for artist_key, genre_list in raw_genres.items():
    # Skip artists that have an empty list [] in the JSON
    if not genre_list: 
        continue
        
    norm_name = normalize_name(artist_key)
    norm_no_the = strip_the_prefix(norm_name)
    
    # Combine the list into a single clean string: "Country, Pop, Rock"
    genre_string = ", ".join(genre_list)
    
    normalized_genres[norm_name] = genre_string
    if norm_no_the:
        normalized_no_the_genres[norm_no_the] = genre_string

print("3. Matching Billboard artists to simplified genres...")
# Initialize new column with empty values
df['Simplified_Genres'] = pd.NA
match_count = 0

for idx, row in df.iterrows():
    listener_name = str(row['listener_artist'])
    norm = normalize_name(listener_name)
    norm_no_the = strip_the_prefix(norm)
    
    # Attempt 1: Exact Normalized Match
    if norm in normalized_genres:
        df.at[idx, 'Simplified_Genres'] = normalized_genres[norm]
        match_count += 1
        
    # Attempt 2: Match without "The"
    elif norm_no_the in normalized_no_the_genres:
        df.at[idx, 'Simplified_Genres'] = normalized_no_the_genres[norm_no_the]
        match_count += 1
        
    # Attempt 3: Split by 'and' or '&' to catch the main artist (e.g. "Perez Prado and His Orchestra" -> "Perez Prado")
    else:
        main_artist_split = re.split(r'\s+and\s+|\s+&\s+', norm)
        if len(main_artist_split) > 1:
            main_artist = main_artist_split[0].strip()
            if main_artist in normalized_genres:
                df.at[idx, 'Simplified_Genres'] = normalized_genres[main_artist]
                match_count += 1

print("4. Filtering to keep ONLY matched artists...")
# Drop any row where we couldn't assign a Simplified Genre
matched_df = df.dropna(subset=['Simplified_Genres']).copy()

# Save the finalized, clean dataset
matched_df.to_csv(OUTPUT_CSV, index=False)

print("\n=== FINAL RESULTS ===")
print(f"Total artists in original Billboard CSV: {len(df)}")
print(f"Successfully matched and saved to new CSV: {len(matched_df)}")
print(f"Artists dropped (No genre found): {len(df) - len(matched_df)}")
print(f"File ready for Gephi network: {OUTPUT_CSV}")

1. Loading files...
2. Prepping JSON genre data...
3. Matching Billboard artists to simplified genres...
4. Filtering to keep ONLY matched artists...

=== FINAL RESULTS ===
Total artists in original Billboard CSV: 4637
Successfully matched and saved to new CSV: 2189
Artists dropped (No genre found): 2448
File ready for Gephi network: data\billboard_to_mbz_with_simplified_genres.csv


In [10]:
import os
import pandas as pd

# --- FILE PATHS ---
DATA_DIR = 'data'
PATH = 'genres'
os.makedirs(PATH, exist_ok=True) # Added this just in case the genres folder doesn't exist yet!

NODES_CSV = os.path.join(DATA_DIR, 'billboard_to_mbz_with_simplified_genres.csv')
EDGES_CSV = os.path.join(DATA_DIR, 'mbz_feature_edges_billboard_only.csv')

# --- CONFIGURATION ---
# STRICT_MODE = True  -> Both Source AND Target must share the genre (Self-contained genre networks)
# STRICT_MODE = False -> Only the Source needs to have the genre (Shows cross-genre collabs)
STRICT_MODE = True 

print("1. Loading datasets...")
nodes_df = pd.read_csv(NODES_CSV).dropna(subset=['mbid', 'Simplified_Genres'])
edges_df = pd.read_csv(EDGES_CSV).dropna(subset=['Source_MBID', 'Target_MBID'])

# Standardize MBIDs to avoid mismatching
nodes_df['mbid'] = nodes_df['mbid'].astype(str).str.strip()
edges_df['Source_MBID'] = edges_df['Source_MBID'].astype(str).str.strip()
edges_df['Target_MBID'] = edges_df['Target_MBID'].astype(str).str.strip()

print("2. Mapping MBIDs to their Genres...")
genre_to_mbids = {}

for _, row in nodes_df.iterrows():
    mbid = row['mbid']
    # Split the "Pop, Rock" strings into lists and strip whitespace
    genres = [g.strip() for g in str(row['Simplified_Genres']).split(',')]
    
    for genre in genres:
        if not genre: 
            continue
        if genre not in genre_to_mbids:
            genre_to_mbids[genre] = set()
        genre_to_mbids[genre].add(mbid)

# Sort by size to show the biggest networks first
sorted_genres = sorted(genre_to_mbids.items(), key=lambda item: len(item[1]), reverse=True)

print(f"\nFound {len(sorted_genres)} unique genres.")
for genre, mbid_set in sorted_genres:
    print(f"  - {genre}: {len(mbid_set)} artists")

# ==========================================
# NEW: GENERATE MASTER "ALL GENRES" NETWORK
# ==========================================
print("\n3. Generating Master 'All Genres' Network CSV...")
all_genred_mbids = set(nodes_df['mbid'])

if STRICT_MODE:
    master_edges = edges_df[
        (edges_df['Source_MBID'].isin(all_genred_mbids)) & 
        (edges_df['Target_MBID'].isin(all_genred_mbids))
    ].copy()
else:
    master_edges = edges_df[edges_df['Source_MBID'].isin(all_genred_mbids)].copy()

master_output_file = os.path.join(PATH, 'network_edges_All_Genres.csv')
if len(master_edges) > 0:
    master_edges.to_csv(master_output_file, index=False)
    print(f" -> Saved [All Genres Master] Network: {len(master_edges)} edges to {master_output_file}")


# ==========================================
# GENERATE INDIVIDUAL GENRE NETWORKS
# ==========================================
print("\n4. Generating Genre-Specific Network CSVs...")
for genre, mbid_set in sorted_genres:
    # Skip weird outlier genres that only have 1 or 2 artists 
    if len(mbid_set) < 5:
        continue
        
    # Clean the genre name so Windows/Mac doesn't crash when creating the file
    safe_genre_name = genre.replace('&', 'n').replace(' ', '_').replace('-', '_')
    output_file = os.path.join(PATH, f'network_edges_{safe_genre_name}.csv')
    
    # Filter the edges
    if STRICT_MODE:
        # Both artists must have this genre
        genre_edges = edges_df[
            (edges_df['Source_MBID'].isin(mbid_set)) & 
            (edges_df['Target_MBID'].isin(mbid_set))
        ].copy()
    else:
        # Only the main artist needs to have this genre
        genre_edges = edges_df[edges_df['Source_MBID'].isin(mbid_set)].copy()
    
    # Save if the network actually has edges
    if len(genre_edges) > 0:
        genre_edges.to_csv(output_file, index=False)
        print(f" -> Saved [{genre}] Network: {len(genre_edges)} edges to {output_file}")
    else:
        print(f" -> Skipped [{genre}] Network: 0 edges found between these artists.")

print("\n--- ALL NETWORKS EXTRACTED SUCCESSFULLY ---")

1. Loading datasets...
2. Mapping MBIDs to their Genres...

Found 7 unique genres.
  - Pop: 1025 artists
  - Rock: 755 artists
  - R&B: 744 artists
  - Country: 558 artists
  - Hip-Hop: 427 artists
  - Jazz: 263 artists
  - Electronic: 229 artists

3. Generating Master 'All Genres' Network CSV...
 -> Saved [All Genres Master] Network: 61569 edges to genres\network_edges_All_Genres.csv

4. Generating Genre-Specific Network CSVs...
 -> Saved [Pop] Network: 13272 edges to genres\network_edges_Pop.csv
 -> Saved [Rock] Network: 5880 edges to genres\network_edges_Rock.csv
 -> Saved [R&B] Network: 10611 edges to genres\network_edges_RnB.csv
 -> Saved [Country] Network: 6537 edges to genres\network_edges_Country.csv
 -> Saved [Hip-Hop] Network: 27323 edges to genres\network_edges_Hip_Hop.csv
 -> Saved [Jazz] Network: 1878 edges to genres\network_edges_Jazz.csv
 -> Saved [Electronic] Network: 3374 edges to genres\network_edges_Electronic.csv

--- ALL NETWORKS EXTRACTED SUCCESSFULLY ---


Creates a network from a CSV:

In [11]:
import os
import pandas as pd
import networkx as nx

def build_billboard_network(top_n, 
                            nodes_file='output/billboard_to_mbz_with_simplified_genres.csv', 
                            edges_file='output_data/FINAL_network_edges.csv',
                            target_genre=None): # <-- NEW PARAMETER
    """
    Builds a self-contained NetworkX Directed Graph using the Top N Billboard artists.
    Optionally filters nodes by a specific genre.
    """
    # 1. Load Nodes
    df_nodes = pd.read_csv(nodes_file).dropna(subset=['mbid', 'artist_mb']).copy()
    df_nodes['mbid'] = df_nodes['mbid'].astype(str).str.strip()
    df_nodes['artist_mb'] = df_nodes['artist_mb'].astype(str).str.strip()
    
    # --- THE FIX: Filter by Genre First ---
    if target_genre:
        # Keep only rows where the target genre exists in the Simplified_Genres column
        df_nodes = df_nodes[df_nodes['Simplified_Genres'].fillna('').str.contains(target_genre, regex=False)]
    
    # Sort by Chart Appearances
    df_nodes['chart_appearances'] = pd.to_numeric(df_nodes['chart_appearances'], errors='coerce').fillna(0)
    df_nodes = df_nodes.sort_values(by='chart_appearances', ascending=False).head(top_n)
    
    seed_mbids = set(df_nodes['mbid'])
    
    # 2. Load Edges
    df_edges = pd.read_csv(edges_file)
    
    # 3. Initialize Graph
    G = nx.DiGraph()
    
    # 4. Add Nodes
    for _, row in df_nodes.iterrows():
        mbid = row['mbid']
        name = row['artist_mb']
        
        G.add_node(
            mbid,
            Label=name,  
            chart_appearances=int(row['chart_appearances']),
            genres=str(row.get('Simplified_Genres', 'Unknown')),
            match_type=str(row.get('match_type', 'Unknown')),
        )
        
    # 5. Filter Edges (Self-Contained)
    df_edges['Source_MBID'] = df_edges['Source_MBID'].astype(str).str.strip()
    df_edges['Target_MBID'] = df_edges['Target_MBID'].astype(str).str.strip()
    
    df_self = df_edges[
        (df_edges['Source_MBID'].isin(seed_mbids)) & 
        (df_edges['Target_MBID'].isin(seed_mbids)) & 
        (df_edges['Source_MBID'] != df_edges['Target_MBID'])
    ].copy()
    
    # 6. Aggregate Repeated Edges into Weights
    edge_weights = (
        df_self.groupby(['Source_MBID', 'Target_MBID'], dropna=False)
        .size()
        .reset_index(name='weight')
    )
    
    for _, row in edge_weights.iterrows():
        G.add_edge(row['Source_MBID'], row['Target_MBID'], weight=int(row['weight']))
        
    return G

In [12]:
import os

# The genres and their specific edge files
genre_files = {
    "All Genres": r"genres\network_edges_All_Genres.csv", # <-- Added the Master Network!
    "Pop": r"genres\network_edges_Pop.csv",          
    "Rock": r"genres\network_edges_Rock.csv",
    "R&B": r"genres\network_edges_RnB.csv",
    "Country": r"genres\network_edges_Country.csv",
    "Hip-Hop": r"genres\network_edges_Hip_Hop.csv",
    "Jazz": r"genres\network_edges_Jazz.csv",
    "Electronic": r"genres\network_edges_Electronic.csv"
}

MASTER_NODES_FILE = r'data\billboard_to_mbz_with_simplified_genres.csv'
genre_graphs = {}

for genre, edge_file in genre_files.items():
    print(f"\n{'='*50}")
    print(f"🎵 BUILDING {genre.upper()} NETWORK")
    print(f"{'='*50}")

    if not os.path.exists(edge_file):
        print(f"[!] Could not find {edge_file}. Skipping...")
        continue

    # --- THE FIX ---
    # If we are building the Master Network, do NOT filter the nodes by a specific genre string
    current_target = None if genre == "All Genres" else genre

    G = build_billboard_network(
        top_n=1000000, 
        nodes_file=MASTER_NODES_FILE, 
        edges_file=edge_file,
        target_genre=current_target # <-- Passes None for the master, and the genre name for the rest
    )
    
    genre_graphs[genre] = G
    print(f"✅ Stored {genre} network in memory. Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")

print("\n🎉 ALL GENRE NETWORKS BUILT AND STORED IN MEMORY!")


🎵 BUILDING ALL GENRES NETWORK
✅ Stored All Genres network in memory. Nodes: 2189 | Edges: 16750

🎵 BUILDING POP NETWORK
✅ Stored Pop network in memory. Nodes: 1025 | Edges: 3751

🎵 BUILDING ROCK NETWORK
✅ Stored Rock network in memory. Nodes: 755 | Edges: 1426

🎵 BUILDING R&B NETWORK
✅ Stored R&B network in memory. Nodes: 744 | Edges: 2948

🎵 BUILDING COUNTRY NETWORK
✅ Stored Country network in memory. Nodes: 558 | Edges: 2152

🎵 BUILDING HIP-HOP NETWORK
✅ Stored Hip-Hop network in memory. Nodes: 427 | Edges: 6499

🎵 BUILDING JAZZ NETWORK
✅ Stored Jazz network in memory. Nodes: 263 | Edges: 364

🎵 BUILDING ELECTRONIC NETWORK
✅ Stored Electronic network in memory. Nodes: 229 | Edges: 671

🎉 ALL GENRE NETWORKS BUILT AND STORED IN MEMORY!


Now we perform Individual Artist Analysis to find the top artists in different genres.

In [13]:
import networkx as nx
import pandas as pd

def compile_all_network_metrics(graphs_dict):
    """
    Takes a dictionary of NetworkX graphs, calculates key centrality metrics 
    for every node, and returns a consolidated Pandas DataFrame.
    """
    all_data = []
    
    for network_name, G in graphs_dict.items():
        print(f"⚙️ Processing '{network_name}' network ({G.number_of_nodes()} nodes)...")
        
        # Skip empty networks just in case
        if G.number_of_nodes() == 0:
            continue

        # 1. Degree Centrality
        in_degree = dict(G.in_degree())
        out_degree = dict(G.out_degree())

        # 2. Eigenvector Centrality
        try:
            eigen = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print(f"  [!] Eigenvector failed to converge for {network_name}. Assigning 0s.")
            eigen = {node: 0 for node in G.nodes()}

        # 3. Betweenness Centrality
        betweenness = nx.betweenness_centrality(G)
        
        # 4. Closeness Centrality
        closeness = nx.closeness_centrality(G)

        # 5. PageRank
        pagerank = nx.pagerank(G, weight='weight')

        # 6. HITS (Hubs and Authorities)
        try:
            hubs, authorities = nx.hits(G, max_iter=1000)
        except nx.PowerIterationFailedConvergence:
            print(f"  [!] HITS failed to converge for {network_name}. Assigning 0s.")
            hubs, authorities = {node: 0 for node in G.nodes()}, {node: 0 for node in G.nodes()}

        # Compile row for every artist in this specific network
        for node in G.nodes():
            row = {
                'Network': network_name,
                'MBID': node,
                'Artist': G.nodes[node].get('Label', node),
                'In_Degree': in_degree.get(node, 0),
                'Out_Degree': out_degree.get(node, 0),
                'Eigenvector': round(eigen.get(node, 0), 6),
                'Betweenness': round(betweenness.get(node, 0), 6),
                'Closeness': round(closeness.get(node, 0), 6),
                'PageRank': round(pagerank.get(node, 0), 6),
                'Authority_Score': round(authorities.get(node, 0), 6),
                'Hub_Score': round(hubs.get(node, 0), 6)
            }
            all_data.append(row)

    print("\n✅ All networks processed! Compiling DataFrame...")
    
    # Create the DataFrame
    df_metrics = pd.DataFrame(all_data)
    
    return df_metrics

In [14]:
df_master_metrics = compile_all_network_metrics(genre_graphs)

⚙️ Processing 'All Genres' network (2189 nodes)...


c:\Users\johng\AppData\Local\Programs\Python\Python311\Lib\site-packages\networkx\algorithms\link_analysis\pagerank_alg.py:453: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.23.5)
  import scipy as sp


ModuleNotFoundError: No module named 'numpy.exceptions'

Creates a nice looking chart of all of the top artists by genre

In [23]:
import pandas as pd

def generate_all_genres_report(df, top_n=5):
    """
    Generates a massive, multi-index summary table containing the top artists 
    for every metric across EVERY genre at the same time.
    """
    print("Building unified master report for all genres...")
    
    metrics = [
        'In_Degree', 'Out_Degree', 'Closeness', 'Betweenness', 
        'Eigenvector', 'PageRank', 'Authority_Score', 'Hub_Score'
    ]
    
    all_rows = []
    
    # Loop through every unique genre in your dataset
    for genre in sorted(df['Network'].unique()):
        sub_df = df[df['Network'] == genre]
        
        for metric in metrics:
            if metric not in sub_df.columns:
                continue
                
            # Get Top N
            top_sorted = sub_df.nlargest(top_n, metric)
            
            # Format as "Artist Name (Score)"
            rank_data = []
            for _, row in top_sorted.iterrows():
                artist = str(row['Artist'])
                score = row[metric]
                
                if isinstance(score, float):
                    rank_data.append(f"{artist} ({score:.4f})")
                else:
                    rank_data.append(f"{artist} ({score})")
                    
            # Pad with blanks just in case a tiny network has fewer than N artists
            while len(rank_data) < top_n:
                rank_data.append("-")
                
            # Build the row dictionary
            row_dict = {
                'Genre': genre,
                'Metric': metric
            }
            for i in range(top_n):
                row_dict[f'Rank {i+1}'] = rank_data[i]
                
            all_rows.append(row_dict)
            
    # Create the final DataFrame
    report_df = pd.DataFrame(all_rows)
    
    # Set a Multi-Index so the table groups nicely by Genre -> Metric
    report_df = report_df.set_index(['Genre', 'Metric'])
    
    # Apply clean styling for Jupyter Notebook
    styled_df = report_df.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'nowrap',
        'padding': '6px 12px',
        'border': '1px solid lightgrey'
    }).set_table_styles([{
        'selector': 'th',
        'props': [('background-color', '#f0f0f0'), ('color', '#333'), ('font-weight', 'bold')]
    }])
    
    return styled_df


In [24]:
display(generate_all_genres_report(df_master_metrics, top_n=5))

# 2. To save this massive table to a CSV (without the HTML styling):
raw_report_df = generate_all_genres_report(df_master_metrics, top_n=5).data
raw_report_df.to_csv("data/ALL_GENRES_TOP_5_REPORT.csv")

Building unified master report for all genres...


Building unified master report for all genres...


Gets and Stores Gephis of all of the genres and the overall network for analysis

In [6]:
import os
import networkx as nx

# 1. Define and create the Gephi export folder
GEPHI_DIR = 'gephi'
os.makedirs(GEPHI_DIR, exist_ok=True)

print(f"🚀 Exporting {len(genre_graphs)} networks to Gephi...")

for genre, G in genre_graphs.items():
    # Create a safe filename (e.g., "R&B" -> "RnB", "All Genres" -> "All_Genres")
    safe_name = genre.replace('&', 'n').replace(' ', '_').replace('-', '_')
    output_path = os.path.join(GEPHI_DIR, f"{safe_name}_network.graphml")
    
    # 2. ⚠️ CRITICAL GEPHI FIX ⚠️
    # Gephi crashes if node attributes are lists, dicts, or None. 
    # We must cast everything to strings or basic numeric types.
    for node, data in G.nodes(data=True):
        for key, value in data.items():
            if value is None:
                G.nodes[node][key] = "Unknown"
            elif not isinstance(value, (int, float, str, bool)):
                G.nodes[node][key] = str(value)
                
    # 3. Export to GraphML
    nx.write_graphml(G, output_path)
    print(f"✅ Exported [{genre}]: {output_path}")

print(f"\n🎉 All done! Your files are waiting in the '{GEPHI_DIR}' folder.")

🚀 Exporting 8 networks to Gephi...
✅ Exported [All Genres]: gephi\All_Genres_network.graphml
✅ Exported [Pop]: gephi\Pop_network.graphml
✅ Exported [Rock]: gephi\Rock_network.graphml
✅ Exported [R&B]: gephi\RnB_network.graphml
✅ Exported [Country]: gephi\Country_network.graphml
✅ Exported [Hip-Hop]: gephi\Hip_Hop_network.graphml
✅ Exported [Jazz]: gephi\Jazz_network.graphml
✅ Exported [Electronic]: gephi\Electronic_network.graphml

🎉 All done! Your files are waiting in the 'gephi' folder.


Now we compare between the genres:

In [ ]:
import networkx as nx
import networkx.algorithms.community as nx_comm
import pandas as pd

def compare_genre_networks(graphs_dict):
    """
    Analyzes the macro-level structure of multiple genre networks 
    and returns a consolidated DataFrame.
    """
    print(f"🌍 Analyzing {len(graphs_dict)} genre networks...")
    
    stats_list = []
    
    for genre, G in graphs_dict.items():
        print(f"  -> Processing {genre}...")
        
        if G.number_of_nodes() == 0:
            continue
            
        # 1. Basic Stats
        nodes = G.number_of_nodes()
        edges = G.number_of_edges()
        
        # Density: Ratio of actual edges to possible edges
        density = nx.density(G)
        
        # Average Degree: Average number of collaborations per artist
        # For directed graphs, Edges / Nodes gives the average out-degree (or in-degree)
        avg_degree = edges / nodes if nodes > 0 else 0
        
        # 2. Clustering Coefficient
        # How tight-knit the "cliques" are
        avg_clustering = nx.average_clustering(G)
        
        # 3. Assortativity
        # Do stars collab with stars (positive), or stars with newcomers (negative)?
        try:
            assortativity = nx.degree_assortativity_coefficient(G)
        except Exception:
            assortativity = None
 
        G_undirected = G.to_undirected()
        
        # 4. Communities (Modularity)
        # How fractured is the genre?
        try:
            communities = list(nx_comm.greedy_modularity_communities(G_undirected))
            num_communities = len(communities)
            # Calculate the actual modularity score (0 = random, >0.3 = strong community structure)
            modularity_score = nx_comm.modularity(G_undirected, communities)
        except Exception:
            num_communities = None
            modularity_score = None
            
        # 5. Average Shortest Path Length
        # Extract the Giant Component first, otherwise isolated nodes will break the math
        try:
            largest_cc = max(nx.connected_components(G_undirected), key=len)
            G_giant = G_undirected.subgraph(largest_cc)
            avg_path_length = nx.average_shortest_path_length(G_giant)
        except Exception:
            avg_path_length = None
            
        # Compile the row
        stats_list.append({
            'Genre': genre,
            'Total Artists (Nodes)': nodes,
            'Total Collabs (Edges)': edges,
            'Density': round(density, 6),
            'Avg Degree (Collabs/Artist)': round(avg_degree, 2),
            'Clustering Coefficient': round(avg_clustering, 4),
            'Assortativity': round(assortativity, 4) if assortativity else None,
            'Sub-Communities': num_communities,
            'Modularity Score': round(modularity_score, 4) if modularity_score else None,
            'Avg Shortest Path Length': round(avg_path_length, 4) if avg_path_length else None
        })

    print("\n✅ Macro-analysis complete!")
    
    # Create DataFrame and sort by number of artists to make it easy to read
    df_stats = pd.DataFrame(stats_list).sort_values(by='Total Artists (Nodes)', ascending=False)
    
    # Reset index for clean formatting
    return df_stats.set_index('Genre')



In [ ]:
# compares the genres across a few different metrics
genre_macro_stats = compare_genre_networks(genre_graphs)
display(genre_macro_stats)

🌍 Analyzing 8 genre networks...
  -> Processing All Genres...
  -> Processing Pop...
  -> Processing Rock...
  -> Processing R&B...
  -> Processing Country...
  -> Processing Hip-Hop...
  -> Processing Jazz...
  -> Processing Electronic...

✅ Macro-analysis complete!


,Total Artists (Nodes),Total Collabs (Edges),Density,Avg Degree (Collabs/Artist),Clustering Coefficient,Assortativity,Sub-Communities,Modularity Score,Avg Shortest Path Length
Genre,,,,,,,,,
All Genres,2189,16750,0.003497,7.65,0.1339,0.2259,605,0.4386,3.6694
Pop,1025,3751,0.003574,3.66,0.1090,0.2559,384,0.4920,3.9954
Rock,755,1426,0.002505,1.89,0.0693,0.1810,362,0.5034,4.1256
R&B,744,2948,0.005333,3.96,0.1111,0.1392,238,0.4517,3.7072
Country,558,2152,0.006924,3.86,0.1168,0.0902,205,0.4639,3.5646
Hip-Hop,427,6499,0.035728,15.22,0.2443,0.0592,49,0.3151,2.4977
Jazz,263,364,0.005283,1.38,0.0562,0.0041,128,0.6136,4.1316
Electronic,229,671,0.012851,2.93,0.1092,0.1206,92,0.4394,3.3184


In [10]:
genre_macro_stats.to_clipboard(sep='\t', index=True)